<p><font size="6" color='grey'> <b>
KI-Agenten. Planen. Handeln. Prüfen.
</b></font> </br></p>



<p><font size="5" color='grey'> <b>
RAG-Konzepte &amp; Embeddings
</b></font> </br></p>

---

**Beitrag zum Leitprojekt:** Ab hier entsteht das **Evidence Tool** des Meeting- & Research-Briefing-Agent. RAG ist keine eigenständige Anwendung, sondern die kontrollierte Fähigkeit, Behauptungen aus Meeting-Protokollen, Entscheidungsvorlagen und Fachartikeln mit abrufbaren Belegen zu unterlegen — Chunking und Embeddings sind die technische Grundlage dafür, dass Mara Vogt Quellen später nachvollziehbar zitiert bekommt. Das ist der erste Baustein der **Handeln**-Fähigkeit im roten Faden des Kurses.

In [14]:
#@title 🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

# LangSmith Env-Vars VOR allen LangChain-Imports setzen
import os
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"]    = "M11-RAG-Konzepte"
os.environ["LANGSMITH_ENDPOINT"]   = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
    show_trace
)

setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# Modell-Konfiguration — Rollen als Konstanten
from genai_lib.model_config import BASELINE, ROUTER, JUDGE, PLANNER, WORKER, WORKER_PREMIUM, CODING, EMBEDDINGS
# LangSmith Tracing
run_cfg = {
    "run_name": "M11_RAG_Konzepte_Embeddings",
    "tags": ["m11", "rag"],
    "metadata": {"notebook": "M11", "version": "1.0"}
}


✓ OPENAI_API_KEY erfolgreich gesetzt
✓ LANGSMITH_API_KEY erfolgreich gesetzt

Python Version: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]

Installierte LangChain- und LangGraph-Bibliotheken:
langchain                                1.4.3
langchain-anthropic                      1.7.5
langchain-chroma                         1.1.0
langchain-classic                        1.0.8
langchain-community                      0.4.2
langchain-core                           1.6.6
langchain-google-genai                   4.4.0
langchain-ollama                         1.1.0
langchain-openai                         1.6.7
langchain-protocol                       0.0.19
langchain-text-splitters                 1.1.3
langgraph                                1.2.11
langgraph-checkpoint                     4.2.0
langgraph-checkpoint-sqlite              3.1.1
langgraph-prebuilt                       1.1.0
langgraph-sdk                            0.4.4

IP-Adresse: 34.186.249.254
Hostname: 254.249.18

In [15]:
#@title 📦 Installationen{ display-mode: "form" }

# install_packages([
#                 ('markitdown[all]', 'markitdown'),
#                 'langchain_huggingface',
#                 ('unstructured[all-docs]>=0.11.2', 'unstructured'),
#                 ])

In [16]:
#@title 📂 Meeting-Briefing-Korpus laden { display-mode: "form" }
import shutil
from pathlib import Path
from genai_lib.utilities import copy_from_github

KORPUS_QUELLE = "ralf-42/Agenten/02_daten/01_text/korpus_meeting_briefing"
KORPUS_MASKE = "*.pdf"
KORPUS_TARGET = "/content/files"

shutil.rmtree(KORPUS_TARGET, ignore_errors=True)
copy_from_github(
    source=KORPUS_QUELLE,
    target=KORPUS_TARGET,
    mask=KORPUS_MASKE,
)

pdfs = sorted(Path(KORPUS_TARGET).glob("*.pdf"))
print(f"Meeting-Briefing-Korpus geladen: {len(pdfs)} PDFs")
for pdf in pdfs[:5]:
    print("-", pdf.name)


  Kopiere: 02_daten/01_text/korpus_meeting_briefing/agenda_vorlage_steuerkreis.pdf → /content/files/agenda_vorlage_steuerkreis.pdf
  Kopiere: 02_daten/01_text/korpus_meeting_briefing/entscheidungsprotokoll_gesamt.pdf → /content/files/entscheidungsprotokoll_gesamt.pdf
  Kopiere: 02_daten/01_text/korpus_meeting_briefing/follow_up_tracker.pdf → /content/files/follow_up_tracker.pdf
  Kopiere: 02_daten/01_text/korpus_meeting_briefing/llm_autonomous_agents_survey_2308_11432.pdf → /content/files/llm_autonomous_agents_survey_2308_11432.pdf
  Kopiere: 02_daten/01_text/korpus_meeting_briefing/projektauftrag_kompass.pdf → /content/files/projektauftrag_kompass.pdf
  Kopiere: 02_daten/01_text/korpus_meeting_briefing/prompt_engineering_survey_2402_07927.pdf → /content/files/prompt_engineering_survey_2402_07927.pdf
  Kopiere: 02_daten/01_text/korpus_meeting_briefing/protokoll_fachbereich_workshop_2026-05-05.pdf → /content/files/protokoll_fachbereich_workshop_2026-05-05.pdf
  Kopiere: 02_daten/01_text

# 1 | Übersicht
---

In [17]:
from genai_lib.utilities import mermaid

# RAG-Architektur
diagram = """
%%{init: {'theme':'forest'}}%%
flowchart TB
    DOC([Dokumente]) --> CHUNK[Chunking]
    CHUNK --> EMBED[Embedding-Modell]
    EMBED --> DB[(Vektordatenbank)]
    QUERY([Nutzeranfrage]) --> QEMBED[Embedding]
    QEMBED --> SEARCH[Similarity Search]
    DB --> SEARCH
    SEARCH --> KONTEXT[Kontext]
    QUERY --> PROMPT[Prompt]
    KONTEXT --> PROMPT
    PROMPT --> LLM[LLM]
    LLM --> ANT([Antwort])
"""
mermaid(diagram, width=700)

**RAG** (**R**etrieval-**A**ugmented **G**eneration) erweitert Sprachmodelle um **externes Wissen**.  
Für Mara bedeutet das: Der Assistant muss nicht aus dem Gedächtnis antworten, sondern kann relevante Stellen aus Meeting-Protokollen, Entscheidungsvorlagen und Fachartikeln abrufen und als Kontext verwenden.

In diesem Modul geht es um die Bausteine dafür:

1. Warum naive Suche nicht reicht
2. Wie Texte in Chunks zerlegt werden
3. Wie Embeddings semantische Nähe messbar machen
4. Wie daraus ein erster Mini-RAG-Loop entsteht



<p><font color='black' size="5">
Vergleich: LLM vs. LLM + RAG
</font></p>

| Eigenschaft | LLM allein | LLM + RAG |
|-------------|-----------|----------|
| Wissensbasis | Bis Trainings-Cutoff | Ständig aktualisierbar |
| Private Daten | ❌ Nicht vorhanden | ✅ Aus Vektordatenbank |
| Halluzinationen | Häufig bei unbekanntem Wissen | Reduziert durch Quellenbindung |
| Anpassbarkeit | Fine-Tuning notwendig | Neue Dokumente einfach hinzufügen |
| Transparenz | Schwer nachvollziehbar | Quellen sichtbar |



# 2 | Warum RAG?
---

<p><font color='black' size="5">
Das Problem: LLMs sind eingefroren
</font></p>


Mara sucht in Meeting-Protokollen und Projektunterlagen nicht nur nach exakten Stichwörtern. Typische Fragen lauten:

- "Welche Entscheidung zur Vektordatenbank gilt aktuell — D1 oder D2?"
- "Welchen Status hat Risiko R2 gerade?"
- "Welche offenen Punkte bestehen laut dem letzten Steuerkreis-Protokoll?"

Eine reine Volltextsuche findet dabei leicht zu viel, zu wenig oder die falschen Stellen. RAG löst dieses Problem, indem semantisch passende Textstellen gesucht und dem LLM als Kontext mitgegeben werden.



<p><font color='black' size="5">
Die RAG-Lösung
</font></p>


RAG trennt das **Reasoning** (im LLM) vom **Wissen** (im Korpus):

1. **Retrieval** – relevante Passagen aus dem Meeting-Briefing-Korpus abrufen
2. **Augmentation** – den Prompt mit diesen Passagen anreichern
3. **Generation** – eine Antwort auf Basis der Quellen erzeugen

**Wann RAG verwenden?**

| Szenario | RAG? | Begründung |
|---|---|---|
| Fachartikel durchsuchen | Ja | Wissen liegt im Korpus, nicht sicher im Modell |
| Allgemeine Smalltalk-Frage | Nein | Modellwissen reicht meist aus |
| Antwort mit Quellenpflicht | Ja | Quellen müssen abrufbar sein |
| Rechenaufgabe | Nein | Ein Tool ist besser geeignet |
| Out-of-Corpus-Frage | Nein | Assistant soll "Nicht im Korpus" sagen |


# 3 | RAG-Architektur
---


<img src="https://raw.githubusercontent.com/ralf-42/Agenten/main/07_image/rag_process.png" width="700" alt="Avatar">
<p><font color='black' size="2">
KI-generiertes Bild
</font></p>

*RAG-Prozess: Von der Indexierung bis zur Antwort-Generierung*


<p><font color='black' size="5">
Einschränkungen von RAG
</font></p>

RAG ist kein Allheilmittel. Es hilft, wenn die Antwort im Korpus belegbar ist. Es hilft nicht, wenn:

- die passenden Dokumente fehlen,
- Retrieval irrelevante Chunks liefert,
- der Prompt Quellenpflicht nicht erzwingt,
- oder das System trotz fehlender Evidenz frei formuliert.

Für den Meeting- & Research-Briefing-Agent ist deshalb wichtig: **Antworten brauchen Quellen oder den Hinweis "Nicht im Korpus".**


<p><font color='darkblue' size="4">
 <b>Viz</b>
</font></p>

- [RAG-Pipeline](https://editor.p5js.org/ralf.bendig.rb/full/RrfB3nCwK)


# 4 | Token-Limits & Context Window
---

Große Sprachmodelle verarbeiten Text als Tokens. Für RAG ist das wichtig, weil Frage, Kontext und Antwort gemeinsam in das Kontextfenster passen müssen.

Wenn zu viele oder zu lange Chunks in den Prompt gelangen, steigen Kosten und Latenz. Wenn zu wenige Chunks verwendet werden, fehlt eventuell die entscheidende Quelle.


**Tokenisierung: Die Grundlage der Textverarbeitung**

Tokenisierung ist der Prozess, bei dem Text in kleinere Einheiten, sogenannte Tokens, zerlegt wird. Diese Tokens können Wörter, Teilwörter oder sogar einzelne Zeichen sein.

**Warum ist Tokenisierung wichtig?**

1. **Einheitliche Verarbeitung:** LLMs haben eine begrenzte Eingabelänge. Die Tokenisierung stellt sicher, dass Texte in einem einheitlichen Format vorliegen, das vom Modell verarbeitet werden kann.

2. **Bedeutungserfassung:** Viele Tokens repräsentieren semantische Einheiten, was dem Modell hilft, die Bedeutung des Textes besser zu erfassen.

3. **Effiziente Verarbeitung:** Tokenisierte Texte können effizienter verarbeitet und in numerische Vektoren umgewandelt werden, was für die Eingabe in neuronale Netze notwendig ist.


**Hilfe:**

1 DIN A4 Seite hat ca. 300 Worte und ca. 450 Token

<p><font color='darkblue' size="4">
 <b>Viz</b>
</font></p>

- [OpenAI Tokenizer](https://platform.openai.com/tokenizer)
- [GPT Tokenizer Playground](https://gpt-tokenizer.dev/)
- [TF Projector](https://projector.tensorflow.org/?hl=de)


Das Kontextfenster muss für **drei Komponenten** reichen:

```
Context Window (z.B. 128k Tokens bei GPT-5.6 Luna)
┌──────────────┬──────────────────┬──────────────┐
│ System-Prompt│  Top-K Chunks    │  Antwort     │
│  ~300 Tokens │  K x 400 Tokens  │  ~500 Tokens │
└──────────────┴──────────────────┴──────────────┘
```

**Praxisregeln**

| Parameter | Empfehlung | Begründung |
|-----------|-----------|------------|
| `chunk_size` | 300–500 Zeichen | Kompakte, fokussierte Chunks |
| `chunk_overlap` | 10–15% von chunk_size | Verhindert Informationsverlust an Grenzen |
| Top-K | 3–5 Chunks | Meist ausreichend, spart Tokens |
| Embedding-Modell | `text-embedding-3-small` | Gutes Preis-Leistungs-Verhältnis |

- Warum Chunk-Größe und Overlap Kosten und Qualität beeinflussen
- Prompt-Budget: Frage + Kontext + Antwortfenster vorausplanen
- Retrieval-Tiefe nur so groß wie nötig

<p><font color='darkblue' size="4">
 <b>Viz</b>
</font></p>

- [Kontextfenster](https://editor.p5js.org/ralf.bendig.rb/full/tLnUgyZRK)



# 5 | Chunking-Strategien
---

<p><font color='black' size="5">
Warum Chunking?
</font></p>

**Chunking: lange Texte in sinnvolle Abschnitte teilen**

Chunking zerlegt Protokolle und Fachartikel in kleinere, zusammenhängende Abschnitte. Für Mara ist das entscheidend: Der Assistant soll nicht ein ganzes Protokoll oder Paper in den Prompt legen, sondern nur die relevanten Passagen.

Gute Chunks sind:

1. klein genug für den Prompt,
2. groß genug für sinnvollen Kontext,
3. mit etwas Overlap gegen Informationsverlust an Grenzen geschützt.


**Strategien im Vergleich**

| Strategie | Vorteil | Nachteil | Einsatz im Kurs |
|---|---|---|---|
| Fixed-Size | Einfach, schnell | Zerreißt Sätze | Nur als Anti-Pattern |
| Satzbasiert | Lesbarer Kontext | Uneinheitliche Länge | Gut für kurze Texte |
| Recursive | Robust und praktikabel | Parameter nötig | Standard für M11-M13 |
| Semantisch | Sehr passend | Teurer und langsamer | Vertiefung in M28 |

**Empfehlung für den Start:** `RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)`


[ChunkViz](https://chunkviz.up.railway.app/)

In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text = (
    "Retrieval-Augmented Generation kombiniert Suche und Antwortgenerierung. "
    "Der Assistant ruft relevante Passagen aus einem Fachartikel-Korpus ab. "
    "Embeddings bilden Text als Vektoren ab. Dadurch können semantisch ähnliche Fragen "
    "und Dokumentstellen gefunden werden, auch bei unterschiedlichen Formulierungen. "
    "Chunking teilt lange Paper in kleinere Abschnitte. So bleibt der Kontext handhabbar "
    "und der Assistant kann Quellen präziser angeben."
)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=220,
    chunk_overlap=40,
    length_function=len,
)

chunks = splitter.split_text(text)

print(f"Original: {len(text)} Zeichen")
print(f"Chunks:   {len(chunks)} Stück (chunk_size=220, overlap=40)")
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ({len(chunk)} Zeichen) ---")
    print(chunk)
    print()


Original: 437 Zeichen
Chunks:   3 Stück (chunk_size=220, overlap=40)
--- Chunk 1 (217 Zeichen) ---
Retrieval-Augmented Generation kombiniert Suche und Antwortgenerierung. Der Assistant ruft relevante Passagen aus einem Fachartikel-Korpus ab. Embeddings bilden Text als Vektoren ab. Dadurch können semantisch ähnliche

--- Chunk 2 (217 Zeichen) ---
ab. Dadurch können semantisch ähnliche Fragen und Dokumentstellen gefunden werden, auch bei unterschiedlichen Formulierungen. Chunking teilt lange Paper in kleinere Abschnitte. So bleibt der Kontext handhabbar und der

--- Chunk 3 (78 Zeichen) ---
bleibt der Kontext handhabbar und der Assistant kann Quellen präziser angeben.



# 6 | Embeddings erklärt
---


<p><font color='black' size="5">
Was sind Embeddings?
</font></p>


Ein **Embedding** ist eine numerische Darstellung von Text als Vektor in einem hochdimensionalen Raum.  
Semantisch ähnliche Texte liegen dabei **nah beieinander** im Vektorraum.

```
"Hund spielt"  →  [0.23, -0.11,  0.87, ...]   <- nah beieinander!
"Tier tobt"    →  [0.21, -0.09,  0.84, ...]
"Aktienkurs"   →  [-0.45, 0.72, -0.31, ...]   <- weit entfernt
```



<p><font color='black' size="5">
OpenAI Embedding-Modelle
</font></p>


| Modell | Dimensionen | Kosten | Empfehlung |
|--------|------------|--------|------------|
| `text-embedding-3-small` | 1.536 | Günstig | ✅ Standard |
| `text-embedding-3-large` | 3.072 | Höher | Anspruchsvolle Suche |
| `text-embedding-ada-002` | 1.536 | Mittel | ⚠️ Veraltet |





| Maß | Wertebereich | Empfehlung |
|-----|------------|------------|
| **Kosinus-Ähnlichkeit** | -1 bis 1 | ✅ OpenAI empfohlen |
| Euklidischer Abstand | 0 bis ∞ | Alternativ |

**Interpretation:** 1.0 = identisch  •  0.5 = verwandt  •  0.0 = unrelated  •  -1.0 = gegensätzlich

<p><font color='darkblue' size="4">
 <b>Viz</b>
</font></p>

- [Embeddings 2D](https://editor.p5js.org/ralf.bendig.rb/full/LPjLkzWbE)
- [Embeddings 3D](https://editor.p5js.org/ralf.bendig.rb/full/gFBwB2E8n)
- [Kontextfenster](https://editor.p5js.org/ralf.bendig.rb/full/tLnUgyZRK)
- [TF Projector](https://projector.tensorflow.org/?hl=de)


# 7 | Embedding-Demo
---

Embeddings für drei Sätze werden erstellt und nach semantischer Ähnlichkeit verglichen:

- **Satz A & B** – Beide über Tiere → sollten **ähnlich** sein
- **Satz A & C** – Tier vs. Börse → sollten **unähnlich** sein

In [19]:
from langchain_openai import OpenAIEmbeddings
import numpy as np

embeddings_model = OpenAIEmbeddings(model=EMBEDDINGS)
print("✅ Embedding-Modell geladen: text-embedding-3-small (1536 Dimensionen)")

✅ Embedding-Modell geladen: text-embedding-3-small (1536 Dimensionen)


In [20]:
# Drei Beispielsätze
satz_a = "Der Hund spielt im Park."
satz_b = "Das Tier tobt auf der Wiese."
satz_c = "Die Aktie fiel gestern um 3 Prozent."

vektoren = embeddings_model.embed_documents([satz_a, satz_b, satz_c])

def cosine_similarity(v1, v2):
    return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))

sim_ab = cosine_similarity(vektoren[0], vektoren[1])
sim_ac = cosine_similarity(vektoren[0], vektoren[2])

mprint(f"""
## Ergebnis: Semantische Ähnlichkeit

| Vergleich | Thema | Ähnlichkeit | Bewertung |
|-----------|-------|-------------|----------|
| A vs. B | Tier-Themen | {sim_ab:10.4f} | 🟢 Ähnlich |
| A vs. C | Tier vs. Börse | {sim_ac:10.4f} | 🔴 Unähnlich |

Vektordimensionen: {len(vektoren[0])}
""")


## Ergebnis: Semantische Ähnlichkeit

| Vergleich | Thema | Ähnlichkeit | Bewertung |
|-----------|-------|-------------|----------|
| A vs. B | Tier-Themen |     0.5441 | 🟢 Ähnlich |
| A vs. C | Tier vs. Börse |     0.2002 | 🔴 Unähnlich |

Vektordimensionen: 1536


# 8 | Mini-RAG: Vom Chunk zur Antwort
---

Die vorherigen Abschnitte haben Embeddings und Chunking einzeln gezeigt.  
Jetzt kombinieren wir beides zu einem **Mini-RAG-Loop**: nicht mit allen PDFs, sondern mit kurzen Meeting-Briefing-Korpus-Auszügen. M12 baut dasselbe Muster anschließend mit ChromaDB und echten Dokumentchunks aus.

**Ablauf:**

1. Kurze Auszüge aus Meeting-/Projekt-PDFs und Fachartikeln werden eingebettet
2. Der Retriever findet die ähnlichsten Auszüge
3. Der Prompt wird mit Kontext angereichert
4. Das LLM antwortet quellengebunden


In [21]:
from langchain.chat_models import init_chat_model
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from genai_lib.model_config import WORKER
# Mini-Wissensdatenbank: kurze, kuratierte Auszüge zum Meeting-Briefing-Korpus.
# M12 ersetzt diese Snippets durch echte PDF-Chunks in ChromaDB.
mini_docs = [
    "Quelle: rag_survey_2312_10997.pdf | RAG kombiniert Retrieval, Augmentation und Generation, um externes Wissen in LLM-Antworten einzubinden.",
    "Quelle: rag_survey_2312_10997.pdf | Naive RAG ist die einfache Grundform; advanced und modular RAG ergänzen Schritte wie Query-Transformation, Re-Ranking und Post-Retrieval-Verarbeitung.",
    "Quelle: ragas_evaluation_2309_15217.pdf | RAG-Evaluation betrachtet unter anderem Antworttreue, Kontextrelevanz und Antwortrelevanz.",
    "Quelle: robust_rag_irrelevant_context_2310_01558.pdf | Irrelevanter Kontext kann RAG-Antworten verschlechtern und ist ein Robustheitsproblem.",
]

vectorstore = Chroma.from_texts(mini_docs, embedding=embeddings_model)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

llm = init_chat_model(WORKER)

rag_prompt = PromptTemplate.from_template("""Beantworte die Frage ausschließlich auf Basis des Kontexts.
Wenn der Kontext nicht reicht, sage: Nicht im Korpus.

Kontext:
{context}

Frage: {question}

Antwort mit Quellenhinweis:""")

mini_rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)
print("Mini-RAG-Chain bereit (in-memory, Research-Snippets)")


Mini-RAG-Chain bereit (in-memory, Research-Snippets)


In [22]:
fragen = [
    "Was sind Bestandteile von RAG?",
    "Wie unterscheidet sich naive RAG von advanced RAG?",
    "Warum kann irrelevanter Kontext problematisch sein?",
]

for frage in fragen:
    antwort = mini_rag_chain.invoke(frage, config=run_cfg)
    retrieved = retriever.invoke(frage, config=run_cfg)
    quellen = sorted({d.page_content.split(" | ")[0].replace("Quelle: ", "") for d in retrieved})
    abgerufen = " | ".join(d.page_content.split(" | ", 1)[1][:80] + "…" for d in retrieved)
    mprint(f"""
---
**Frage:** {frage}
**Quellen:** {", ".join(quellen)}
**Abgerufen:** {abgerufen}
**Antwort:** {antwort}
""")



---
**Frage:** Was sind Bestandteile von RAG?
**Quellen:** rag_survey_2312_10997.pdf
**Antwort:** Bestandteile von RAG sind laut Kontext die **einfache Grundform (Naive RAG)** sowie bei **advanced/modular RAG** zusätzliche Schritte wie **Query-Transformation**, **Re-Ranking** und **Post-Retrieval-Verarbeitung**.  
**Quelle:** rag_survey_2312_10997.pdf



---
**Frage:** Wie unterscheidet sich naive RAG von advanced RAG?
**Quellen:** rag_survey_2312_10997.pdf
**Antwort:** Naive RAG ist die einfache Grundform; advanced RAG ergänzt Schritte wie Query-Transformation, Re-Ranking und Post-Retrieval-Verarbeitung.  
**Quelle:** rag_survey_2312_10997.pdf



---
**Frage:** Warum kann irrelevanter Kontext problematisch sein?
**Quellen:** robust_rag_irrelevant_context_2310_01558.pdf
**Antwort:** Irrelevanter Kontext kann problematisch sein, weil er RAG-Antworten verschlechtern kann und damit ein Robustheitsproblem darstellt. Quelle: robust_rag_irrelevant_context_2310_01558.pdf


In [23]:
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t; _t.sleep(5)
show_trace("M11-RAG-Konzepte", limit=3, show_steps=True)

## LangSmith Trace — `M11-RAG-Konzepte`

| Run | Status | Dauer | Child-Runs |
|-----|--------|-------|------------|
| `M11_RAG_Konzepte_Embeddings` | ❌ pending | — | 3 |
| `M11_RAG_Konzepte_Embeddings` | ✅ success | 1.1s | 4 |
| `M11_RAG_Konzepte_Embeddings` | ✅ success | 1.2s | 4 |


### Steps — letzter Run: `M11_RAG_Konzepte_Embeddings`

| # | Typ | Name | Status | Dauer |
|---|-----|------|--------|-------|
| 1 | `chain` | `RunnableParallel<context,question>` | ✅ | 0.3s |
| 2 | `prompt` | `PromptTemplate` | ✅ | 0.0s |
| 3 | `llm` | `ChatOpenAI` | ✅ | 1.0s |
| 4 | `parser` | `StrOutputParser` | ✅ | 0.0s |

# A | Aufgaben
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabenstellungen bieten Anregungen; alternative Herausforderungen sind möglich.

**Hinweis zur Lösungshilfe:**
> Generative KI kann als Unterstützung beim Lernen und Entwickeln genutzt werden. Bei Fehlermeldungen, Teilproblemen oder Code-Varianten kann zum Beispiel Gemini in Google Colab helfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.


**Grundlagen**
Erstelle Embeddings für einen kleinen Korpus mit **6-9 Sätzen aus 3 Themen**: RAG-Grundlagen, RAG-Evaluation und Robustheit (wie in den Mini-RAG-Auszügen aus Kapitel 8).

- `embeddings_model.embed_documents([...])` auf alle Sätze anwenden.
- Die Funktion `cosine_similarity(v1, v2)` aus Kapitel 7 verwenden.
- Kosinus-Ähnlichkeiten für mindestens 3 Satzpaare berechnen.
- Vektoren in `meine_vektoren` und Werte in `sim_ergebnisse` speichern.

**✅ Erledigt wenn:** `sim_ergebnisse` enthält mindestens 3 Kosinus-Ähnlichkeiten und alle Werte liegen zwischen -1 und 1.


In [24]:
# Grundlagen: Embeddings + Kosinus-Ähnlichkeit
# Startpunkt: embeddings_model und cosine_similarity(v1, v2) aus Kapitel 7

# 1. 6-9 Sätze aus 3 Themen definieren (RAG-Grundlagen, RAG-Evaluation, Robustheit)
# 2. embeddings_model.embed_documents() auf alle Sätze anwenden, Vektoren in meine_vektoren speichern
# 3. cosine_similarity() für mindestens 3 Satzpaare berechnen, Werte in sim_ergebnisse sammeln


**Aufbau**
Untersuche, ob Embeddings Sätze desselben Themas näher zusammenlegen als Sätze verschiedener Themen.

- Die Sätze aus den Grundlagen nach Themen gruppieren (z. B. `rag_grundlagen`, `evaluation`, `robustheit`).
- Ähnlichkeiten innerhalb eines Themas (intra) und zwischen Themen (inter) vergleichen.
- Die Ergebnisse als Tabelle ausgeben: Paar, Typ, Ähnlichkeit.
- Prüfen: Sind Sätze zu RAG-Grundlagen untereinander ähnlicher als zu Robustheit oder Evaluation?

**✅ Erledigt wenn:** `aufbau_ergebnisse` enthält mindestens zwei intra- und zwei inter-Themen-Vergleiche; die Tabelle zeigt Paar, Typ und Ähnlichkeit.


In [25]:
# Aufbau: Intra- vs. Inter-Thema-Vergleich
# Startpunkt: embeddings_model, cosine_similarity() und die Sätze aus den Grundlagen

# 1. Sätze nach Themen gruppieren (saetze_nach_thema, z. B. rag_grundlagen, evaluation, robustheit)
# 2. Je Thema Embeddings berechnen (thema_vektoren)
# 3. Intra-Thema- und Inter-Thema-Kosinus-Ähnlichkeiten berechnen
# 4. Ergebnisse als Liste von Dicts (paar, typ, aehnlichkeit) in aufbau_ergebnisse sammeln und als Tabelle ausgeben


**Vertiefung**
Vergleiche `text-embedding-3-small` mit `text-embedding-3-large` anhand eines Satzpaars zu RAG-Evaluation.

- Ein zweites Modell `embed_large` erstellen und dieselbe Ähnlichkeit wie mit `embeddings_model` (small) berechnen.
- Beurteilen, ob sich die semantische Nähe sichtbar verändert.
- In 2-3 Sätzen erklären, wann sich der höhere Aufwand von `-large` für den Meeting- & Research-Briefing-Agent lohnt (Kapitel 6).

**✅ Erledigt wenn:** `sim_small` und `sim_large` enthalten die Ähnlichkeit für mindestens 1 Satzpaar; `embedding_diskussion` enthält mindestens 2 Sätze.


In [26]:
# Vertiefung: small vs. large Embedding-Modelle
# Startpunkt: embeddings_model (small) und cosine_similarity() aus Kapitel 7

# 1. embed_large = OpenAIEmbeddings(model="text-embedding-3-large")
# 2. Ein Satzpaar zu RAG-Evaluation mit beiden Modellen einbetten, Ähnlichkeit berechnen (sim_small, sim_large)
# 3. In embedding_diskussion (>= 2 Sätze) einordnen, wann sich -large für den Meeting- & Research-Briefing-Agent lohnt


# B | Dokumente zum Weiterlesen
---

Ergänzende Artikel aus der Kurs-Dokumentation:

- [RAG-Konzepte](https://ralf-42.github.io/Agenten/04-agenten-implementierung/kontext-wissen/rag-konzepte.html)
- [Embeddings](https://ralf-42.github.io/Agenten/04-agenten-implementierung/kontext-wissen/embeddings.html)
- [Tokenizing & Chunking](https://ralf-42.github.io/Agenten/04-agenten-implementierung/kontext-wissen/tokenizing-chunking.html)
- [Context Engineering](https://ralf-42.github.io/Agenten/04-agenten-implementierung/kontext-wissen/context-engineering.html)
- [Einsteiger ChromaDB](https://ralf-42.github.io/Agenten/05-frameworks/einsteiger-chromadb.html)
